# 10.1 幂法与反幂法

特征值问题寻找满足 $A x = \lambda x$ 的标量 $\lambda$ 和非零向量 $x$。本节从 Rayleigh 商和残差出发，比较幂法、带位移反幂法和 Rayleigh 商迭代。

In [ ]:
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(parent for parent in pathlib.Path.cwd().parents if (parent / 'src').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    eigen_residual_norm,
    inverse_power_method,
    power_method,
    rayleigh_quotient,
    rayleigh_quotient_iteration,
)

## Rayleigh 商和特征残差

若 $x$ 已经接近某个特征向量，Rayleigh 商 $\rho(x)=x^T A x / x^T x$ 给出自然的特征值估计。但相邻估计变化很小并不一定说明特征对可信，因此还要观察 $\|A x - \rho(x)x\|_2$。

In [ ]:
A = np.array([
    [4.0, 1.0, 0.0],
    [1.0, 3.0, 1.0],
    [0.0, 1.0, 2.0],
])
x = np.array([1.0, 1.0, 0.0])

rq = rayleigh_quotient(A, x)
residual = eigen_residual_norm(A, rq, x / np.linalg.norm(x))
rq, residual

## 幂法

幂法不断计算 $x_{k+1}=A x_k/\|A x_k\|$。当主特征值在模意义下唯一，且初始向量含有主特征向量分量时，迭代会逐步对准主特征向量。

In [ ]:
dominant = power_method(A, initial=[1.0, 0.5, 0.25], tolerance=1e-12)
dominant.eigenvalue, dominant.iterations, dominant.residual_norm

In [ ]:
np.column_stack([dominant.eigenvalue_history, dominant.residual_history])[:8]

## 带位移反幂法

反幂法对 $(A-\sigma I)y_k=x_k$ 求解线性系统，然后归一化。离位移 $\sigma$ 最近的特征值会在变换矩阵 $(A-\sigma I)^{-1}$ 中变成模最大的特征值。

In [ ]:
near_middle = inverse_power_method(A, shift=2.8, initial=[1.0, 1.0, 1.0], tolerance=1e-12)
near_middle.eigenvalue, near_middle.iterations, near_middle.residual_norm

## Rayleigh 商迭代

Rayleigh 商迭代把反幂法的位移改成当前 Rayleigh 商。对称矩阵和足够好的初值下，它通常比固定 shift 的反幂法更快。

In [ ]:
rqi = rayleigh_quotient_iteration(A, initial=[0.2, 1.0, 0.4], tolerance=1e-13)
rqi.eigenvalue, rqi.iterations, rqi.residual_norm

In [ ]:
np.linalg.eigvalsh(A)

## 小结

* Rayleigh 商给出特征值估计，特征残差衡量特征对质量。
* 幂法偏向模最大的特征值。
* 反幂法通过位移选择目标特征值。
* Rayleigh 商迭代自动更新位移，适合对称矩阵的局部快速求精。